In [ ]:
import os
import shutil
import tarfile
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
import pandas as pd
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go
from wordcloud import WordCloud, STOPWORDS
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from keras.callbacks import Callback
from keras.optimizers import Adam
from datetime import datetime

# Used to save the model
class SaveModelAndTokenizerCallback(Callback):
    def __init__(self, model, tokenizer, save_path):
        super(SaveModelAndTokenizerCallback, self).__init__()
        self.model = model
        self.tokenizer = tokenizer
        self.save_path = save_path

    def on_epoch_end(self, epoch, logs=None):
        epoch_save_path = os.path.join(self.save_path, f'epoch_{epoch + 1}')
        os.makedirs(epoch_save_path, exist_ok=True)
        tokenizer_save_path = os.path.join(epoch_save_path, 'Tokenizer')
        model_save_path = os.path.join(epoch_save_path, 'Model')

        # Save tokenizer
        self.tokenizer.save_pretrained(tokenizer_save_path)
        # Save model
        self.model.save_pretrained(model_save_path)
        print(f"Saved model and tokenizer at epoch {epoch + 1}")

# Define the path to save the models
save_path = '/content/drive/MyDrive/BERT4'


# load preprocessed data from drive and create dataframes
with open('/content/drive/MyDrive/train_pos_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

with open('/content/drive/MyDrive/test_cleaned.txt', 'r', encoding='utf-8') as file:
    test_content = file.readlines()

train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1

train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0

test_df = pd.DataFrame(test_content, columns=['tweet'])

train = pd.concat([train_pos, train_neg], ignore_index = True)

tweets = train['tweet']

labels = train['label']

tests = test_df['tweet']


# Split training and validation set
train_tweets, val_tweets, train_labels, val_labels = train_test_split(tweets, labels, test_size=0.1, random_state=42)

# Use bert(base) tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Uncomment this if load saved tokenizer from drive
#tokenizer = BertTokenizer.from_pretrained('/content/drive/MyDrive/BERT3/epoch_1/Tokenizer')

max_len= 40

X_train_encoded = tokenizer.batch_encode_plus(train_tweets.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

X_val_encoded = tokenizer.batch_encode_plus(val_tweets.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

X_test_encoded = tokenizer.batch_encode_plus(tests.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

# Use Bert(base) model
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Uncomment this if load saved model from drive
#model = TFBertForSequenceClassification.from_pretrained('/content/drive/MyDrive/BERT3/epoch_1/Model')

model.dropout = tf.keras.layers.Dropout(0.1)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# Save model and tokenizer after each epoch
save_callback = SaveModelAndTokenizerCallback(model, tokenizer, save_path)

print(model.summary())
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


history = model.fit(
    [X_train_encoded['input_ids'], X_train_encoded['token_type_ids'], X_train_encoded['attention_mask']],
    train_labels,
    validation_data=(
      [X_val_encoded['input_ids'], X_val_encoded['token_type_ids'], X_val_encoded['attention_mask']],val_labels),
    batch_size=256,
    epochs=5,
    callbacks=[save_callback]
)